## Objectives
- look at correlations
- Use log function to transform the data
- Handle the duplicates
- Handle missing values
- Standardize and normalize the data
- Handle the outliers

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pylab as plt # matplotlib inline
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from scipy.stats import norm
from scipy import stats

#### Retrieving data via API call

In [ ]:
import requests

# url to the data
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/Ames_Housing_Data1.tsv"
filename = "Ames_Housing_Data.tsv"
response = requests.get(url)
response

In [ ]:
# write content into a file
with open(filename, 'wb') as f:
    f.write(response.content)

In [ ]:
# read the data 
housing = pd.read_csv(filename, sep='\t')
housing.head()

## Correlations
- Pair plots
- Scatter plots
- Heat maps
- Correlation matrices
`.corr()` function is based on pearson correlation coefficient. It can only be measured on the numerical attributes

In [ ]:
# select only numeric attributes
housing_num = housing.select_dtypes(include=['float64', 'int64'])
housing_num_corr = housing_num.corr()
housing_num_corr

In [ ]:
# now only look at sale price
housing_num_corr = housing_num.corr()['SalePrice']
housing_num_corr

In [ ]:
# select features with corr. coef more than 0.5 - strong correlation
top_features = housing_num_corr[abs(housing_num_corr)>0.5].sort_values(ascending=False)
top_features

In [ ]:
# use pair plots to visualize the correlation between features and target (SalePrice) 
# sns.pairplot(housing_num, x_vars=housing_num.columns, y_vars=['SalePrice'])
for i in range(0, len(housing_num.columns), 5):
    sns.pairplot(data=housing_num, x_vars=housing_num.columns[i:i+5], y_vars=['SalePrice'])

## Log Transformation
Some ML models assume the target is normally distributed. We can check this assumption using `sns.displot`

In [ ]:
sns.displot(housing['SalePrice']);

## Skewness 
Longer tail on the right side - positive skew. This skewness is the measure of assymetry of the distibution and we can check it using `.skew()` function 

In [ ]:
housing['SalePrice'].skew()

In the fairly distributed data the range of skewness is between -0.5 and 0.5, moderately distributed data has skewness between -0.5 and -1.0, or 0.5 and 1.0; highly skewed distribution is <-1.0 and 1.0<. In our case the skewness is 1.7 that means the distribution is highly skewed.

To make our data normally distributed we have several options
- log transform - `np.log()`
- square root transform - `np.sqrt()`
- Box-Cox transform - `scipy.stats.boxcox`

In [ ]:
log_transformed=np.log(housing['SalePrice'])
sp_transformed=sns.displot(log_transformed)
print('Skewness is %f' % log_transformed.skew()) # now skewness should be well within the range

# Exercises

### Handling Duplicates
- First check whether our data contains any duplicates(hint: `.duplicated()` or `index.is_unique`)
- Then remove duplicates

### Handling missing values
- First detect missing values (`.isna()`, `.isnull()` etc.)
- Use `.sort_values()` to find worst offenders i.e most missing values
- Then handle those missing values: use `.dropna()`, `.drop()`, `.fillna()` and explain trade-offs

### Handling outliers
- Find outliers. You can use box plot or scatter plot. first use uni-variate analysis (using one variable) then use multi-variate analysis (two or more variables)
- With your own words explain how box plot works in markdown cell
- Would you keep or remove those outliers?
- Then perform bi-variate analysis of the two features: 'SalePrice' and 'GrLivArea', plot the scatter plot
- Find the indices of outliers appear in scatter plot and delete them using `.drop()`, then plot the scatter plot again
- Do the same with 'Lot Area' feature


### Feature Scaling
- When the values are in complete different scales (e.g. age and income) we perform feature scaling
- There are 2 ways of doing this:normalization (min-max scaling) and standardization. In normalization the data is rescaled so that it end up in the range [0,1]. In standardization the resulting distribution has zero variance. They are also calculated in different ways. Normalization: 
$\frac{x- x_{min}}{x_{max}-x_{min}}$ Standardization: $\frac{x-x_{mean}}{x_{std}}$ deviation. The functions we need are `MinMaxScalar` and `StandardScalar` from scikit-learn library
- But first try to write these functions yourself using numpy or pandas then you can import from scikit-learn
- print statistics before scaling and after scaling, explain the differences